# Reference: Tessera Pipeline Data Structures

# Overview

The Tessera pipeline produces six intermediate data structures, one per stage:

```
make_cells()    →  cells
make_mesh()     →  mesh
compute_field() →  field
compute_morse() →  mesh$morse   (assigned back onto mesh)
run_dmt()       →  dmt_run
make_tiles()    →  tiles
```

This vignette runs each stage on a small example dataset and inspects every
slot so you can see exactly what each object contains and how they relate to
one another.

# Libs and data

In [ ]:
suppressPackageStartupMessages(library(tessera))

data('tessera_warmup')
counts    = tessera_warmup$counts
meta_data = tessera_warmup$meta_data
cat('Input: ', nrow(meta_data), 'cells,', nrow(counts), 'genes\n')

In [ ]:
## Parameters used throughout
npcs                  = 10
prune_thresh_quantile = 0.95
prune_min_cells       = 10
smooth_distance       = 'projected'
smooth_similarity     = 'projected'
max_npts              = 50
min_npts              = 5
alpha                 = 1

---

# `cells` — raw inputs {#cells}

`cells` is the canonical input container.  It carries coordinates, counts,
and metadata, and gains embeddings after `make_mesh()` prunes outlier cells.

In [ ]:
coords = matrix(
    c(meta_data$X, meta_data$Y), ncol = 2,
    dimnames = list(NULL, c('x', 'y'))
)
cells = make_cells(coords, NULL, counts, data.table::as.data.table(meta_data))
names(cells)

## `cells$coords`

N × 2 numeric matrix. One row per input cell.

In [ ]:
dim(cells$coords)
head(cells$coords)

## `cells$embeddings`

Named list of N × D matrices. `NULL` at construction; set to
`list(pca = ...)` after pruning (see [`make_mesh()`]).

In [ ]:
cells$embeddings    # NULL until we call do_pca below

## `cells$counts`

G × N `dgCMatrix` of raw counts.

In [ ]:
dim(cells$counts)   # genes × cells
cells$counts[1:4, 1:4]

## `cells$meta`

`data.table` with N rows of per-cell metadata.

In [ ]:
dim(cells$meta)
head(cells$meta)

---

# `mesh` — Delaunay triangulation {#mesh}

`make_mesh()` triangulates cell coordinates, prunes long edges and small
connected components, and adds degenerate exterior triangles along the boundary.

In [ ]:
mesh = make_mesh(cells, list(
    prune_thresh_quantile = prune_thresh_quantile,
    prune_mincells        = prune_min_cells
))
names(mesh)

## `mesh$pts`

Data.table of surviving cells — fewer rows than `cells$coords` when any
cells are pruned.

In [ ]:
dim(mesh$pts)
names(mesh$pts)
head(mesh$pts)

`ORIG_ID` is a 1-based index into the original `cells$coords` and `cells$meta`:

In [ ]:
## Look up the cell type of the first 6 surviving cells
cells$meta$type[mesh$pts$ORIG_ID[1:6]]

## `mesh$edges`

One row per edge of the triangulation. Every edge has a **primal** form
(connecting two points) and a **dual** form (connecting the centroids of its
two flanking triangles).

In [ ]:
dim(mesh$edges)
names(mesh$edges)

| Column | Description |
|--------|-------------|
| `from_pt`, `to_pt` | Row indices into `mesh$pts` |
| `from_tri`, `to_tri` | Row indices into `mesh$triangles` |
| `x0_pt`, `y0_pt`, `x1_pt`, `y1_pt` | Primal endpoint coordinates |
| `x0_tri`, `y0_tri`, `x1_tri`, `y1_tri` | Dual endpoint coordinates |
| `boundary` | `TRUE` for edges on the convex-hull boundary |

In [ ]:
table(mesh$edges$boundary)

## `mesh$triangles`

One row per triangle, including degenerate exterior triangles.

In [ ]:
dim(mesh$triangles)
names(mesh$triangles)
table(mesh$triangles$external)

| Column | Description |
|--------|-------------|
| `X`, `Y` | Triangle centroid |
| `area` | Triangle area |
| `height` | Circumradius (used for pruning) |
| `external` | `TRUE` for degenerate boundary triangles |

## `mesh$tri_to_pt`

Sparse incidence matrix (num_tris × num_pts). Entry (i, j) = 1 if triangle
i contains point j. Used to propagate point-level gradients up to triangles.

In [ ]:
dim(mesh$tri_to_pt)   # num_tris × num_pts

## `mesh$adj`

Symmetric adjacency matrix (num_pts × num_pts). Entry (i, j) = 1 if points
i and j share an edge. Used for gradient estimation and bilateral smoothing.

In [ ]:
dim(mesh$adj)         # num_pts × num_pts

## `mesh$morse`

`NULL` here; populated by `compute_morse()` below.

In [ ]:
mesh$morse

---

# `field` — spatial gradient field {#field}

`compute_field()` estimates the transcriptional gradient at every mesh element
and SVD-compresses it to six scalars.

**Important:** compute PCA on the *pruned* cells (`mesh$pts$ORIG_ID`), not
the full input, then assign to `cells$embeddings`:

In [ ]:
pca              = do_pca(cells$counts[, mesh$pts$ORIG_ID, drop = FALSE], npcs)
cells$embeddings = list(pca = pca$embeddings)
dim(cells$embeddings$pca)   # num_surviving_cells × npcs

In [ ]:
field = compute_field(cells, mesh, list(
    smooth_distance   = smooth_distance,
    smooth_similarity = smooth_similarity
))
names(field)
names(field$gradient)

## SVD gradient columns

Every gradient matrix has the same six columns:

In [ ]:
colnames(field$gradient$pts)

| Column | Description |
|--------|-------------|
| `dx_grad`, `dy_grad` | Principal direction of transcriptional change |
| `dx_ortho`, `dy_ortho` | Direction orthogonal to gradient (isovalue direction) |
| `len_grad` | Gradient magnitude — large near tissue boundaries |
| `len_ortho` | Orthogonal singular value |

`len_grad + len_ortho` becomes the scalar field `f` used by DMT.

## Gradient matrices — dimensions

| Slot | Rows | Content |
|------|------|---------|
| `gradient$pts` | num_pts | One gradient per surviving cell |
| `gradient$tris` | num_tris | Weighted average of vertex gradients |
| `gradient$edges_pts` | num_edges | Primal edge gradient |
| `gradient$edges_tris` | num_edges | Dual edge gradient |

In [ ]:
dim(field$gradient$pts)
dim(field$gradient$tris)
dim(field$gradient$edges_pts)
dim(field$gradient$edges_tris)

In [ ]:
head(field$gradient$pts)

---

# `mesh$morse` — Discrete Morse Theory {#morse}

`compute_morse()` derives a scalar field from gradient magnitudes, builds two
spanning forests (primal on points, dual on triangles), and traces the
separatrix edges that define tile boundaries.

In [ ]:
mesh$morse = compute_morse(field, mesh)
names(mesh$morse)

## `mesh$morse$f` — scalar field values

In [ ]:
names(mesh$morse$f)

| Slot | Length | Description |
|------|--------|-------------|
| `f$pts` | num_pts | Boundary score at each cell |
| `f$tris` | num_tris | Boundary score at each triangle |
| `f$edges_prim` | num_edges | Primal edge boundary score |
| `f$edges_dual` | num_edges | Dual edge boundary score |

In [ ]:
length(mesh$morse$f$pts)    # one value per surviving cell
length(mesh$morse$f$tris)   # one value per triangle
summary(mesh$morse$f$pts)

## `mesh$morse$prim` — primal minimum spanning forest

Built on points. Its minima are the seed cells, one per initial tile.

In [ ]:
names(mesh$morse$prim)
length(mesh$morse$prim$minima)   # number of initial tile seeds
head(mesh$morse$prim$edges)      # tree edges: x0,y0 → x1,y1

## `mesh$morse$dual` — dual maximum spanning forest

Built on triangles. Its maxima are the "ridge" triangles that sit at the
peaks of the boundary score field, flanking the separatrices.

In [ ]:
names(mesh$morse$dual)
length(mesh$morse$dual$maxima)   # number of ridge triangles

## `mesh$morse$e_sep` — separatrix edge indices

Integer vector of 1-based indices into `mesh$edges`. These edges run along
tissue boundaries and are removed by `run_dmt()` to partition cells into
initial tiles.

In [ ]:
length(mesh$morse$e_sep)
## Separatrix edges as a fraction of all edges:
length(mesh$morse$e_sep) / nrow(mesh$edges)
## A few separatrix edges:
head(mesh$edges[mesh$morse$e_sep, .(from_pt, to_pt, x0_tri, y0_tri, x1_tri, y1_tri)])

---

# `dmt_run` — initial tile assignments {#dmt_run}

`run_dmt()` removes the separatrix edges from the mesh graph and labels each
resulting connected component as a distinct initial tile.

In [ ]:
dmt_run = run_dmt(mesh)
names(dmt_run)

## `dmt_run$pts`

Copy of `mesh$pts` with one added column:

In [ ]:
names(dmt_run$pts)
head(dmt_run$pts)

| Column | Description |
|--------|-------------|
| `agg_id` | Integer initial tile ID |

In [ ]:
cat('Surviving cells: ', nrow(dmt_run$pts), '\n')
cat('Initial tiles:   ', length(unique(dmt_run$pts$agg_id)), '\n')

The initial tiles are fine-grained — many will be merged in `make_tiles()`.

## `dmt_run$edges`

Copy of `mesh$edges` with two added columns:

In [ ]:
setdiff(names(dmt_run$edges), names(mesh$edges))
head(dmt_run$edges[, .(from_pt, to_pt, agg_from, agg_to)])

| Column | Description |
|--------|-------------|
| `agg_from` | Tile ID of the `from_pt` endpoint |
| `agg_to` | Tile ID of the `to_pt` endpoint |

---

# `tiles` — final merged tiles {#tiles}

`make_tiles()` agglomeratively merges initial tiles into final spatial tiles,
pooling counts and computing spatial summaries for each.

In [ ]:
tiles = make_tiles(cells, mesh, dmt_run, list(
    alpha    = alpha,
    max_npts = max_npts,
    min_npts = min_npts
))
names(tiles)

## `tiles$meta_data` — one row per tile

In [ ]:
names(tiles$meta_data)
head(dplyr::select(tiles$meta_data, -shape))

| Column | Description |
|--------|-------------|
| `id` | Tile identifier |
| `X`, `Y` | Tile centroid |
| `npts` | Number of constituent cells |
| `shape` | `sf` MULTIPOLYGON tile boundary |
| `area`, `perimeter` | Tile geometry summaries |

In [ ]:
cat('Final tiles:', nrow(tiles$meta_data), '\n')
summary(tiles$meta_data$npts)

`shape` is an `sf` geometry column:

In [ ]:
class(tiles$meta_data$shape)
tiles$meta_data$shape[1]

## `tiles$counts` — pooled gene counts

Genes × num_tiles sparse matrix. Column names match `tiles$meta_data$id`.

In [ ]:
dim(tiles$counts)
identical(colnames(tiles$counts), tiles$meta_data$id)
tiles$counts[1:4, 1:4]

## `tiles$pcs` — per-tile PCA embeddings

num_tiles × D matrix. Each row is the mean of the constituent cells'
PC embeddings. Row names match `tiles$meta_data$id`.

In [ ]:
dim(tiles$pcs)
head(tiles$pcs[, 1:4])

## `tiles$adj` — tile adjacency matrix

Symmetric dgCMatrix (num_tiles × num_tiles). Entry (i, j) = 1 if tiles i
and j share a border.

In [ ]:
dim(tiles$adj)
mean(Matrix::colSums(tiles$adj > 0))   # average number of tile neighbours

## `tiles$edges` — inter-tile edges

Data.table with one row per pair of adjacent tiles.

In [ ]:
dim(tiles$edges)
names(tiles$edges)
head(dplyr::select(tiles$edges, from, to, npts, edge_length))

## `tiles$cell_ids` — cell-to-tile mapping

The primary way to transfer tile labels back to individual cells.

In [ ]:
head(tiles$cell_ids)
cat('Cells assigned to tiles:', nrow(tiles$cell_ids), 'of', nrow(meta_data), '\n')

| Column | Description |
|--------|-------------|
| `ORIG_ID` | 1-based index into original `cells$coords` / `meta_data` |
| `tile_id` | Tile identifier (matches `tiles$meta_data$id`) |

---

# Cross-referencing objects {#cross-ref}

## Transfer tile labels to original cells

In [ ]:
meta_data$tile_id = NA
meta_data$tile_id[tiles$cell_ids$ORIG_ID] = tiles$cell_ids$tile_id
table(is.na(meta_data$tile_id))   # FALSE = assigned, TRUE = pruned as outlier

## All cells in a given tile

In [ ]:
first_tile = tiles$meta_data$id[1]
cells_in_tile1 = tiles$cell_ids[tile_id == first_tile, ORIG_ID]
meta_data[cells_in_tile1, ]

## Cells that survived mesh pruning

In [ ]:
## mesh$pts$ORIG_ID indexes into the original meta_data
surviving_meta = meta_data[mesh$pts$ORIG_ID, ]
cat('Surviving cells:', nrow(surviving_meta), 'of', nrow(meta_data), '\n')

## Object dimension summary

In [ ]:
cat(sprintf(
    'Input cells:          %d\nSurviving (pruned):   %d\nInitial tiles (DMT):  %d\nFinal tiles:          %d\n',
    nrow(meta_data),
    nrow(mesh$pts),
    length(unique(dmt_run$pts$agg_id)),
    nrow(tiles$meta_data)
))

---

# Session Info

In [ ]:
sessionInfo()